## Clicker Mechanic: Secret Coin System

All three Gate Game levels use a **Clicker** object for bonus coins. Players find and click hidden objects to earn extra coins beyond the level's normal collectibles.

**The Secret:**
- Each level has ONE clickable object (coin, chest, or power core)
- Click it multiple times to earn 1 bonus coin
- Each level has different click requirements and coin caps
- A **yellow counter** shows your progress on the object

**Level Comparison:**

| Level | Object | Clicks/Coin | Max Coins | Location |
|-------|--------|-------------|-----------|----------|
| Cannon Ball | Blinking Coin | 3 | 40 | Moves randomly |
| Escape Room | Treasure Chest | 3 | 15 | Hidden bottom-left |
| Zone Catch | Power Core | 4 | 20 | Top center |

## How The Clicker Actually Works

### Step-by-Step Process:

**1. You Click The Object**
- Mouse click triggers `handleClick()` on the Clicker object
- Click counter (`this.clcks`) increases by 1
- Yellow number on object updates to show new count

**2. Check If You Earned A Coin**
- The `interact()` function checks: `clicks % CLICKS_PER_COIN === 0`
- Example (Cannon Ball): `9 % 3 === 0` ✓ (3rd, 6th, 9th clicks give coins)
- Example (Zone Catch): `12 % 4 === 0` ✓ (4th, 8th, 12th clicks give coins)

**3. Coin Is Added (If Under Cap)**
- Checks if you've reached max bonus coins (`_bonusCoinsGiven < MAX_BONUS`)
- If under cap: adds 1 to `gameEnv.stats.coinsCollected`
- Increments `_bonusCoinsGiven` to track total bonus coins from this object

**4. Visual Feedback**
- Counter keeps incrementing even after cap (shows total clicks)
- But coins stop being added once MAX_BONUS reached
- In Cannon Ball: counter resets when coin repositions

## Clicker Implementation Code

The **Clicker** class extends **Npc** and handles all click counting and coin rewards:

{% highlight javascript %}
// Clicker.js - The core click handler
class Clicker extends Npc {
    constructor(data = {}, gameEnv = null) {
        super(data, gameEnv);
        this.clcks = 0;  // Tracks total clicks on this object
    }

    handleClick(event) {
        if (this.interact) {
            this.clcks++;  // Increment click counter
            // Call the interact callback with click count
            this.interact(this.clcks, this.spriteData?.id, this);
        }
    }

    draw() {
        super.draw();  // Draw the sprite image
        
        // Draw click counter overlay (yellow text)
        if (this.visible && this.drawCounter) {
            this.ctx.font = 'bold 32px Arial';
            this.ctx.fillStyle = 'yellow';
            this.ctx.textAlign = 'center';
            
            // Center the counter on the object
            const centerX = this.canvas.width / 2;
            const centerY = this.canvas.height / 2;
            
            // Black outline + yellow fill
            this.ctx.strokeStyle = 'black';
            this.ctx.lineWidth = 4;
            this.ctx.strokeText(this.clcks.toString(), centerX, centerY);
            this.ctx.fillText(this.clcks.toString(), centerX, centerY);
        }
    }
}
{% endhighlight %}

## Cannon Ball Level - Blinking Coin Clicker

**How The Secret Works:**
1. A coin appears and blinks somewhere in the arena
2. Click it 3 times → Get 1 bonus coin (modulo 3)
3. Every 2.8 seconds: coin disappears, repositions, reappears
4. Click counter RESETS when coin moves (starts over at 0)
5. Maximum 40 bonus coins total from this clicker
6. This is the ONLY coin source in Cannon Ball level

**Example Clicking Session:**
- Click 1, 2 → Counter shows "2" (no coin yet)
- Click 3 → Counter shows "3", **+1 COIN** added to stats
- Click 4, 5 → Counter shows "4", "5"
- Click 6 → Counter shows "6", **+1 COIN** added
- [Coin repositions] → Counter resets to 0
- Continue until 40 bonus coins earned, then no more rewards

In [ ]:
%%js

// GAME_RUNNER: Cannon Ball - Blinking Coin Clicker | hide_edit: true, width: 100%, height: 720px, isolated: true
// Self-contained runner: keep HUD / leaderboard / coin counters inside the game frame.
// This runner should not spill into the surrounding blog post layout.

import GameControl from '@assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameLevelCannonball from '@assets/js/projects/gategame/levels/GameLevelCannonball.js';

export const gameLevelClasses = [GameLevelCannonball];
export { GameControl };


**Cannon Ball Clicker Setup Code:**
{% highlight javascript %}
const clickerData = {
    id: 'BonusChest',
    src: 'images/gamebuilder/sprites/coin.png',
    SCALE_FACTOR: 30,  // Larger size for easier clicking
    INIT_POSITION: { x: 0.45, y: 0.30 },
    
    // THIS is where the coin-giving magic happens
    interact: function (clicks) {
        const CLICKS_PER_COIN = 3;  // Need 3 clicks per bonus coin
        const MAX_BONUS = 40;        // Maximum 40 bonus coins
        
        // Initialize bonus counter on first click ever
        if (typeof this._bonusCoinsGiven !== 'number') {
            this._bonusCoinsGiven = 0;
        }
        
        // Stop giving coins if we hit the cap
        if (this._bonusCoinsGiven >= MAX_BONUS) return;
        
        // Check if this click number is divisible by 3
        // clicks=3: 3%3=0 ✓ GIVE COIN
        // clicks=6: 6%3=0 ✓ GIVE COIN
        // clicks=9: 9%3=0 ✓ GIVE COIN
        if (clicks % CLICKS_PER_COIN === 0) {
            // Add 1 coin to the game stats
            gameEnv.stats.coinsCollected = (gameEnv.stats.coinsCollected || 0) + 1;
            // Track that we gave a bonus coin
            this._bonusCoinsGiven++;
        }
    }
};

this.classes = [
    { class: Clicker, data: clickerData }
];
{% endhighlight %}

## Escape Room Level - Hidden Treasure Chest

**How The Secret Works:**
1. Treasure chest is hidden in bottom-left corner behind dungeon fog
2. You must explore the maze to find it
3. Once found, click it 3 times → Get 1 bonus coin (modulo 3)
4. Chest stays in same position (doesn't move like Cannon Ball coin)
5. Click counter stays on screen, keeps counting up
6. Maximum 15 bonus coins total from this chest
7. Level also has 9 regular walk-over coins scattered in the dungeon

**Example Clicking Session:**
- [Find chest in corner]
- Click 1, 2 → Counter shows "2" (no coin yet)
- Click 3 → Counter shows "3", **+1 COIN** added
- Click 4, 5 → Counter shows "4", "5"
- Click 6 → Counter shows "6", **+1 COIN** added
- Continue clicking until 15 bonus coins earned
- After 15 bonus coins: clicking still increments counter but no more coins

In [ ]:
%%js

// GAME_RUNNER: Escape Room - Hidden Treasure Chest | hide_edit: true, width: 100%, height: 720px, isolated: true
// Self-contained runner: keep all game details inside the runner frame.
// This runner should not spill into the surrounding blog post layout.

import GameControl from '@assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameLevelEscaperoom from '@assets/js/projects/gategame/levels/GameLevelEscaperoom.js';

export const gameLevelClasses = [GameLevelEscaperoom];
export { GameControl };


**Escape Room Clicker Setup Code:**
{% highlight javascript %}
const treasureClickerData = {
    id: 'HiddenTreasure',
    src: path + "/images/gamebuilder/sprites/mastergate.png",
    SCALE_FACTOR: 22,
    INIT_POSITION: { x: 340, y: 620 },  // Tucked in bottom-left
    zIndex: 400,
    
    // THIS is where the coin-giving magic happens
    interact: function (clicks) {
        const CLICKS_PER_COIN = 3;  // Need 3 clicks per bonus coin
        const MAX_BONUS = 15;        // Maximum 15 bonus coins
        
        // Initialize bonus counter on first click ever
        if (typeof this._bonusCoinsGiven !== 'number') {
            this._bonusCoinsGiven = 0;
        }
        
        // Stop giving coins if we hit the cap
        if (this._bonusCoinsGiven >= MAX_BONUS) return;
        
        // Check if this click number is divisible by 3
        // clicks=3: 3%3=0 ✓ GIVE COIN
        // clicks=6: 6%3=0 ✓ GIVE COIN  
        // clicks=45: 45%3=0 ✓ GIVE COIN (15th bonus coin, then stops)
        if (clicks % CLICKS_PER_COIN === 0) {
            // Add 1 coin to the game stats
            gameEnv.stats.coinsCollected = (gameEnv.stats.coinsCollected || 0) + 1;
            // Track that we gave a bonus coin
            this._bonusCoinsGiven++;
        }
    }
};

this.classes = [
    { class: Clicker, data: treasureClickerData }
];
{% endhighlight %}

## Zone Catch Level - Power Core Clicker

**How The Secret Works:**
1. Power core floats at top-center, always visible
2. Click it 4 times → Get 1 bonus coin (modulo 4) **[HARDER RATIO]**
3. Core stays in position while you play Zone Catch mini-game
4. You must multitask: match colors AND click power core
5. Click counter stays visible, keeps counting up
6. Maximum 20 bonus coins total from this core
7. Level also has 3 regular walk-over coins at start

**Example Clicking Session:**
- Click 1, 2, 3 → Counter shows "3" (no coin yet - need 4!)
- Click 4 → Counter shows "4", **+1 COIN** added
- Click 5, 6, 7 → Counter shows "7"
- Click 8 → Counter shows "8", **+1 COIN** added
- Continue until 20 bonus coins earned (requires 80 total clicks)
- After 20 bonus coins: clicking still increments counter but no more coins

In [ ]:
%%js

// GAME_RUNNER: Zone Catch - Power Core Clicker | hide_edit: true, width: 100%, height: 720px, isolated: true
// Self-contained runner: keep HUD / leaderboard / coin counters inside the game frame.
// This runner should not spill into the surrounding blog post layout.

import GameControl from '@assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameLevelZonecatch from '@assets/js/projects/gategame/levels/GameLevelZonecatch.js';

export const gameLevelClasses = [GameLevelZonecatch];
export { GameControl };


**Zone Catch Clicker Setup Code:**
{% highlight javascript %}
const powerCoreClickerData = {
    id: 'PowerCore',
    src: path + "/images/gamebuilder/sprites/mastergate.png",
    SCALE_FACTOR: 18,
    INIT_POSITION: { 
        x: Math.round(width * 0.5) - 64,   // Center horizontally
        y: Math.round(height * 0.12)       // 12% from top
    },
    zIndex: 400,
    
    // THIS is where the coin-giving magic happens
    interact: function (clicks) {
        const CLICKS_PER_COIN = 4;  // Need 4 clicks per bonus coin (HARDER!)
        const MAX_BONUS = 20;        // Maximum 20 bonus coins
        
        // Initialize bonus counter on first click ever
        if (typeof this._bonusCoinsGiven !== 'number') {
            this._bonusCoinsGiven = 0;
        }
        
        // Stop giving coins if we hit the cap
        if (this._bonusCoinsGiven >= MAX_BONUS) return;
        
        // Check if this click number is divisible by 4
        // clicks=4: 4%4=0 ✓ GIVE COIN
        // clicks=8: 8%4=0 ✓ GIVE COIN
        // clicks=80: 80%4=0 ✓ GIVE COIN (20th bonus coin, then stops)
        if (clicks % CLICKS_PER_COIN === 0) {
            // Add 1 coin to the game stats
            gameEnv.stats.coinsCollected = (gameEnv.stats.coinsCollected || 0) + 1;
            // Track that we gave a bonus coin
            this._bonusCoinsGiven++;
        }
    }
};

this.classes = [
    { class: Clicker, data: powerCoreClickerData }
];
{% endhighlight %}

## Summary: The Clicker Secret Revealed

**The Core Mechanic (Same For All Levels):**

1. **Find The Clickable Object**
   - Cannon Ball: Blinking coin (moves every 2.8s)
   - Escape Room: Treasure chest (hidden bottom-left)
   - Zone Catch: Power core (visible top-center)

2. **Click It Multiple Times**
   - Each click increments the counter (yellow number)
   - The `interact()` function checks the click count

3. **Modulo Check Determines Coin Reward**
   - `clicks % CLICKS_PER_COIN === 0` → Give coin
   - Cannon Ball & Escape Room: Every 3rd click (3, 6, 9, 12...)
   - Zone Catch: Every 4th click (4, 8, 12, 16...)

4. **Cap Prevents Infinite Coins**
   - `_bonusCoinsGiven` tracks total bonus coins from this clicker
   - Once MAX_BONUS reached, no more coins (but counter keeps going)

5. **Visual Feedback**
   - Yellow counter with black outline
   - Shows total clicks on this object
   - Helps player know when next coin is coming

**Total Clicks Required:**
- Cannon Ball: 120 clicks for all 40 bonus coins (40 × 3)
- Escape Room: 45 clicks for all 15 bonus coins (15 × 3)
- Zone Catch: 80 clicks for all 20 bonus coins (20 × 4)

## Runner Isolation Update

The game runner cells below are set up to stay self-contained so the HUD, leaderboard, and coin counters stay inside each runner instead of spilling into the blog post layout.


## Setup Instructions

**Quick Start:**
1. Clone repo with collaborator access
2. Run `make dev` in terminal
3. Open `localhost:4500/gamify`
4. Test the clicker secret in each level

**Documentation:** [Gamify README](https://pages.opencodingsociety.com/gamify/overview)